# Notebook 2: Territory Integrity & Optimization Readiness Assessment

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:

import pandas as pd
import geopandas as gpd
import json

from shapely.geometry import shape
from shapely.validation import explain_validity

orders = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Twiga Data Science/twiga_routing_model/orders.csv')
shops = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Twiga Data Science/twiga_routing_model/shops.csv')
zones = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Twiga Data Science/twiga_routing_model/zones.csv')
depots = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Twiga Data Science/twiga_routing_model/depots-master.csv')
routes = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Twiga Data Science/twiga_routing_model/routes.csv')
vehicles = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Twiga Data Science/twiga_routing_model/trucks-vehicles.csv')
dispatch = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Twiga Data Science/twiga_routing_model/dispatch-jan-april.csv')

print("Data Loaded Successfully")


Data Loaded Successfully


## Create GeoDataFrames

In [4]:

def parse_polygon(x):
    try:
        return shape(json.loads(x))
    except Exception:
        return None

zones['geometry'] = zones['geographic_bounds'].apply(parse_polygon)

zone_gdf = gpd.GeoDataFrame(zones, geometry='geometry', crs='EPSG:4326')

shop_gdf = gpd.GeoDataFrame(
    shops,
    geometry=gpd.points_from_xy(shops.longitude, shops.latitude),
    crs='EPSG:4326'
)


## Geometry Health Check

In [5]:

print("Total Zones:", len(zone_gdf))
print("Valid Geometries:", zone_gdf.geometry.notna().sum())
print("Invalid Geometries:", zone_gdf.geometry.isna().sum())


Total Zones: 101
Valid Geometries: 101
Invalid Geometries: 0


## Polygon Validity

In [6]:

zone_gdf['is_valid'] = zone_gdf.geometry.is_valid
zone_gdf['is_valid'].value_counts()


,count
is_valid,
True,84
False,17


## Invalid Polygon Details

In [7]:

invalid_polygons = zone_gdf[zone_gdf['is_valid'] == False].copy()

invalid_polygons['validation_error'] = (
    invalid_polygons.geometry.apply(explain_validity)
)

invalid_polygons[['zone_id','name','validation_error']]


,zone_id,name,validation_error
9,019c8ae2-4868-77f9-952c-102da411e8ce,DZ-1,Self-intersection[36.8735605568853 -1.32725947...
15,019e4697-b8bc-7670-ade4-d3baf1385480,DZ-52,Self-intersection[36.8735201006427 -1.32720653...
17,019e4697-c2c8-7d81-959e-441705044647,DZ-53,Self-intersection[36.8330211577189 -1.26688499...
19,019e4697-e0b4-7290-bd1e-d3cc5ef4c252,DZ-56,Self-intersection[36.9021229367744 -1.28212527...
20,019c8ae2-4867-7c64-a1eb-8292c839ca3e,DZ-9,Self-intersection[36.5741133344782 -1.17274241...
28,019c8ae2-4869-703f-8e26-8d9c80b10bd8,DZ-4,Self-intersection[36.941621420505 -1.115254119...
37,019c8ae2-4867-7741-ac40-bb622fa43bbd,DZ-10,Self-intersection[36.7799618393522 -1.23080599...
43,c81df03d-c692-53a7-8682-c1067d2782cf,DZ-8,Ring Self-intersection[36.965521 -1.235801]
50,019e4697-f4ac-71dc-915b-9102865a503a,DZ-58,Self-intersection[36.8872213733358 -1.30103635...
52,019e4698-6d16-7a1d-8738-38008d792924,DZ-70,Self-intersection[36.9378795776521 -1.40459303...


## Coverage Analysis

In [8]:

shop_zone = gpd.sjoin(
    shop_gdf,
    zone_gdf[['zone_id','geometry']],
    how='left',
    predicate='within'
)

coverage = shop_zone['zone_id'].notna().mean()
print(f"Coverage: {coverage:.2%}")


Coverage: 97.71%


## Unzoned Shops

In [9]:

unzoned = shop_zone[shop_zone['zone_id'].isna()]

print("Unzoned Shops:", len(unzoned))
unzoned.head()


Unzoned Shops: 746


,shop_id,shop_name,latitude,longitude,route_id,date_created,geometry,index_right,zone_id
0,b6f75a09-967e-4ea6-bc52-c3558e315a2a,Mama Roy Shop Riat,-0.040402,34.761505,14fb3f97-caa7-45ff-a17d-5e1f81baeed0,2026-06-05 11:02:59.248000+00:00,POINT (34.7615 -0.0404),NaN,NaN
2,465b2a49-1137-48fc-b34d-124fd33fb9b0,Kamama Ness Shop Kianja,-0.040113,34.807056,14fb3f97-caa7-45ff-a17d-5e1f81baeed0,2026-06-05 06:33:07.356000+00:00,POINT (34.80706 -0.04011),NaN,NaN
5,dc6ba0dc-7202-4e8b-9596-5048da01444b,Mama Sharon's Shop vispa,-0.115891,34.786392,594f7579-5cae-40e9-967e-73867d5ad334,2026-06-03 11:20:34.996000+00:00,POINT (34.78639 -0.11589),NaN,NaN
19,fa134ea0-d9e5-440a-ace1-e157053c6354,Skyroll shop Usenge,-0.064453,34.061275,2fb98e42-97da-4707-b9cc-a84ce53c39f9,2026-05-29 14:53:50.466000+00:00,POINT (34.06128 -0.06445),NaN,NaN
23,65e3cbec-272b-45f2-97ec-2baac5a7a6bc,Likoni connectors shop Bondo,-0.094725,34.258051,e3c3490f-0611-4974-8e88-56929196a3d7,2026-05-05 13:01:24.920000+00:00,POINT (34.25805 -0.09472),NaN,NaN


## Overlap Severity

In [10]:

shop_zone_all = gpd.sjoin(
    shop_gdf,
    zone_gdf[['zone_id','geometry']],
    how='inner',
    predicate='within'
)

shop_overlap_counts = (
    shop_zone_all.groupby('shop_id').size()
)

shop_overlap_counts.describe()


,0
count,6735.000000
mean,4.723682
std,1.608596
min,1.000000
25%,3.000000
50%,5.000000
75%,6.000000
max,8.000000


## Zone Overlap Matrix

In [11]:

overlap_results = []

for i, row1 in zone_gdf.iterrows():
    for j, row2 in zone_gdf.iterrows():
        if i >= j:
            continue

        try:
            if row1.geometry.intersects(row2.geometry):
                overlap_results.append({
                    'zone_a': row1['name'],
                    'zone_b': row2['name']
                })
        except:
            pass

overlaps = pd.DataFrame(overlap_results)

print(overlaps.shape)
overlaps.head()


(960, 2)


,zone_a,zone_b
0,Limuru+Limuru 2+Ruaka 1+Ruaka 2,DZ-36
1,Limuru+Limuru 2+Ruaka 1+Ruaka 2,DZ-34
2,Limuru+Limuru 2+Ruaka 1+Ruaka 2,Gachie
3,Limuru+Limuru 2+Ruaka 1+Ruaka 2,DZ-74
4,Limuru+Limuru 2+Ruaka 1+Ruaka 2,DZ-33


## Depot Pressure Index

In [12]:

orders_enriched = orders.merge(
    depots,
    on='depot_id',
    how='left'
)

depot_pressure = (
    orders_enriched
    .groupby('depot_name')['weight']
    .agg(
        orders='count',
        total_weight='sum',
        avg_weight='mean'
    )
    .sort_values('total_weight', ascending=False)
)

depot_pressure


,orders,total_weight,avg_weight
depot_name,,,
Embakasi,1861,92785.210,49.857716
Syokimau,1653,81779.964,49.473662
West Nairobi,1342,70057.294,52.203647
Nairobi Central,1482,63730.909,43.003312
Kiambu,1471,61844.868,42.042738
Eastlands,1025,55555.599,54.200584
Kahawa,1081,50603.986,46.812198
Thika Town,727,39853.906,54.819678
Rongai,728,35946.887,49.377592


## Vehicle Capacity

In [13]:

vehicles['load_capacity'] = pd.to_numeric(
    vehicles['load_capacity'],
    errors='coerce'
)

vehicle_capacity = (
    vehicles.groupby('depot_id')['load_capacity']
    .sum()
    .reset_index()
)

vehicle_capacity.head()


,depot_id,load_capacity
0,21ee323a-6149-4310-908e-74a72cd6cb09,236700.0
1,287570cb-a2d0-4bb8-bede-11ad462bc65a,16000.0
2,2da4f36b-0634-4616-8649-c74f1b7f9ae6,258250.0
3,3259d09a-a825-4e53-ab3f-657d1c415eb0,59000.0
4,36aab551-1b63-4794-a827-d58b95d16556,70800.0


## Kenya Boundary Sanity Check

In [14]:

zone_gdf['centroid_x'] = zone_gdf.geometry.centroid.x
zone_gdf['centroid_y'] = zone_gdf.geometry.centroid.y

outside_kenya = zone_gdf[
    (zone_gdf.centroid_x < 33) |
    (zone_gdf.centroid_x > 42) |
    (zone_gdf.centroid_y < -5) |
    (zone_gdf.centroid_y > 5)
]

outside_kenya[['zone_id','name','centroid_x','centroid_y']]


/tmp/ipykernel_4069/238140958.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  zone_gdf['centroid_x'] = zone_gdf.geometry.centroid.x
/tmp/ipykernel_4069/238140958.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  zone_gdf['centroid_y'] = zone_gdf.geometry.centroid.y


,zone_id,name,centroid_x,centroid_y


## Territory Readiness Report

In [15]:

coverage_pct = shop_zone['zone_id'].notna().mean()

overlap_pct = (
    len(shop_overlap_counts[shop_overlap_counts > 1])
    / len(shop_gdf)
)

print("="*50)
print("TERRITORY READINESS REPORT")
print("="*50)
print(f"Coverage Rate: {coverage_pct:.2%}")
print(f"Overlap Rate: {overlap_pct:.2%}")
print("="*50)


TERRITORY READINESS REPORT
Coverage Rate: 97.71%
Overlap Rate: 89.11%
